<a href="https://colab.research.google.com/github/Chikka-Pradhayani/ABTalks-60-Days-AI-Challenge/blob/main/Day-38.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sqlite3

conn = sqlite3.connect("feedback.db")

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS feedback (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    session_id TEXT NOT NULL,
    query TEXT NOT NULL,
    response_excerpt TEXT NOT NULL,
    rating INTEGER NOT NULL CHECK (rating IN (1, -1)),
    timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
)
""")

conn.commit()

print("Feedback database and table created successfully!")

conn.close()

Feedback database and table created successfully!


In [2]:
import sqlite3

conn = sqlite3.connect("feedback.db")
cursor = conn.cursor()

cursor.execute("PRAGMA table_info(feedback)")
columns = cursor.fetchall()

for column in columns:
    print(column)

conn.close()

(0, 'id', 'INTEGER', 0, None, 1)
(1, 'session_id', 'TEXT', 1, None, 0)
(2, 'query', 'TEXT', 1, None, 0)
(3, 'response_excerpt', 'TEXT', 1, None, 0)
(4, 'rating', 'INTEGER', 1, None, 0)
(5, 'timestamp', 'DATETIME', 0, 'CURRENT_TIMESTAMP', 0)


In [3]:
import sqlite3

conn = sqlite3.connect("feedback.db")
cursor = conn.cursor()

cursor.execute("""
INSERT INTO feedback (session_id, query, response_excerpt, rating)
VALUES (?, ?, ?, ?)
""", (
    "test-session-001",
    "What is machine learning?",
    "Machine learning is a method where computers learn patterns from data.",
    1
))

conn.commit()

print("Test feedback inserted successfully!")

conn.close()

Test feedback inserted successfully!


In [4]:
import sqlite3

conn = sqlite3.connect("feedback.db")
cursor = conn.cursor()

cursor.execute("SELECT * FROM feedback")

records = cursor.fetchall()

for record in records:
    print(record)

conn.close()

(1, 'test-session-001', 'What is machine learning?', 'Machine learning is a method where computers learn patterns from data.', 1, '2026-09-20 08:11:51')


In [5]:
from fastapi import FastAPI
from pydantic import BaseModel
import sqlite3

app = FastAPI()

class Feedback(BaseModel):
    session_id: str
    query: str
    response_excerpt: str
    rating: int

@app.post("/feedback")
def submit_feedback(feedback: Feedback):
    conn = sqlite3.connect("feedback.db")
    cursor = conn.cursor()

    cursor.execute("""
    INSERT INTO feedback
    (session_id, query, response_excerpt, rating)
    VALUES (?, ?, ?, ?)
    """, (
        feedback.session_id,
        feedback.query,
        feedback.response_excerpt[:100],
        feedback.rating
    ))

    conn.commit()
    conn.close()

    return {"message": "Feedback received successfully"}

In [6]:
from collections import Counter
import re

@app.get("/feedback/summary")
def feedback_summary():
    conn = sqlite3.connect("feedback.db")
    cursor = conn.cursor()

    cursor.execute("SELECT query, rating, response_excerpt FROM feedback")
    records = cursor.fetchall()

    conn.close()

    total_interactions = len(records)

    positive_count = sum(1 for record in records if record[1] == 1)

    positive_percentage = (
        (positive_count / total_interactions) * 100
        if total_interactions > 0
        else 0
    )

    words = []

    stop_words = {
        "the", "is", "a", "an", "and", "or", "to", "of",
        "in", "on", "for", "what", "how", "why", "are",
        "was", "were", "can", "i", "you", "my", "it"
    }

    for query, rating, response in records:
        query_words = re.findall(r"\b[a-zA-Z]{3,}\b", query.lower())

        for word in query_words:
            if word not in stop_words:
                words.append(word)

    top_topic_words = [
        word for word, count in Counter(words).most_common(5)
    ]

    lowest_rated = [
        {
            "response_excerpt": record[2],
            "rating": record[1]
        }
        for record in records
        if record[1] == -1
    ][0:3]

    return {
        "total_interactions": total_interactions,
        "positive_rating_percentage": round(positive_percentage, 2),
        "top_5_query_topic_words": top_topic_words,
        "three_lowest_rated_response_excerpts": lowest_rated
    }

In [8]:
import threading
import uvicorn

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("FastAPI server started on port 8000")

FastAPI server started on port 8000


In [9]:
import requests

feedback_data = {
    "session_id": "test-session-002",
    "query": "Explain Python loops",
    "response_excerpt": "A loop allows you to repeat a block of code multiple times.",
    "rating": 1
}

response = requests.post(
    "http://127.0.0.1:8000/feedback",
    json=feedback_data
)

print("Status code:", response.status_code)
print("Response:", response.json())

INFO:     127.0.0.1:45368 - "POST /feedback HTTP/1.1" 200 OK
Status code: 200
Response: {'message': 'Feedback received successfully'}


In [10]:
import requests

response = requests.get("http://127.0.0.1:8000/feedback/summary")

print("Status code:", response.status_code)
print("Summary:")
print(response.json())

INFO:     127.0.0.1:52664 - "GET /feedback/summary HTTP/1.1" 200 OK
Status code: 200
Summary:
{'total_interactions': 2, 'positive_rating_percentage': 100.0, 'top_5_query_topic_words': ['machine', 'learning', 'explain', 'python', 'loops'], 'three_lowest_rated_response_excerpts': []}


In [11]:
feedback_data = {
    "session_id": "test-session-003",
    "query": "How do I deploy my FastAPI application?",
    "response_excerpt": "You can deploy FastAPI using a cloud hosting platform.",
    "rating": -1
}

response = requests.post(
    "http://127.0.0.1:8000/feedback",
    json=feedback_data
)

print("Status code:", response.status_code)
print("Response:", response.json())

INFO:     127.0.0.1:34112 - "POST /feedback HTTP/1.1" 200 OK
Status code: 200
Response: {'message': 'Feedback received successfully'}


In [12]:
response = requests.get("http://127.0.0.1:8000/feedback/summary")

print(response.json())

INFO:     127.0.0.1:60202 - "GET /feedback/summary HTTP/1.1" 200 OK
{'total_interactions': 3, 'positive_rating_percentage': 66.67, 'top_5_query_topic_words': ['machine', 'learning', 'explain', 'python', 'loops'], 'three_lowest_rated_response_excerpts': [{'response_excerpt': 'You can deploy FastAPI using a cloud hosting platform.', 'rating': -1}]}


In [13]:
import sqlite3

conn = sqlite3.connect("feedback.db")
cursor = conn.cursor()

cursor.execute("""
SELECT id, session_id, query, rating, timestamp
FROM feedback
ORDER BY id
""")

records = cursor.fetchall()

for record in records:
    print(record)

conn.close()

(1, 'test-session-001', 'What is machine learning?', 1, '2026-09-20 08:11:51')
(2, 'test-session-002', 'Explain Python loops', 1, '2026-09-20 08:14:22')
(3, 'test-session-003', 'How do I deploy my FastAPI application?', -1, '2026-09-20 08:14:56')


In [14]:
!pip install pyngrok

In [15]:
from google.colab import output

public_url = output.eval_js(
    "google.colab.kernel.proxyPort(8000)"
)

print(public_url)

https://8000-m-s-kkb-use1c2-1y919o1ac5g0m-c.us-east1-2.prod.colab.dev


In [16]:
import sqlite3

conn = sqlite3.connect("feedback.db")
cursor = conn.cursor()

cursor.execute("""
SELECT session_id, query, response_excerpt, rating, timestamp
FROM feedback
ORDER BY id DESC
""")

for record in cursor.fetchall():
    print(record)

conn.close()

('test-session-003', 'How do I deploy my FastAPI application?', 'You can deploy FastAPI using a cloud hosting platform.', -1, '2026-09-20 08:14:56')
('test-session-002', 'Explain Python loops', 'A loop allows you to repeat a block of code multiple times.', 1, '2026-09-20 08:14:22')
('test-session-001', 'What is machine learning?', 'Machine learning is a method where computers learn patterns from data.', 1, '2026-09-20 08:11:51')


In [22]:
import sqlite3

conn = sqlite3.connect("feedback.db")
cursor = conn.cursor()

cursor.execute("""
SELECT id, session_id, query, response_excerpt, rating, timestamp
FROM feedback
ORDER BY id DESC
""")

for record in cursor.fetchall():
    print(record)

conn.close()

(3, 'test-session-003', 'How do I deploy my FastAPI application?', 'You can deploy FastAPI using a cloud hosting platform.', -1, '2026-09-20 08:14:56')
(2, 'test-session-002', 'Explain Python loops', 'A loop allows you to repeat a block of code multiple times.', 1, '2026-09-20 08:14:22')
(1, 'test-session-001', 'What is machine learning?', 'Machine learning is a method where computers learn patterns from data.', 1, '2026-09-20 08:11:51')


In [23]:
import requests

response = requests.get("http://127.0.0.1:8000/feedback/summary")

print("Status code:", response.status_code)
print("Summary:", response.json())

INFO:     127.0.0.1:59628 - "GET /feedback/summary HTTP/1.1" 200 OK
Status code: 200
Summary: {'total_interactions': 3, 'positive_rating_percentage': 66.67, 'top_5_query_topic_words': ['machine', 'learning', 'explain', 'python', 'loops'], 'three_lowest_rated_response_excerpts': [{'response_excerpt': 'You can deploy FastAPI using a cloud hosting platform.', 'rating': -1}]}


In [26]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

new_app = FastAPI()

new_app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

print("New FastAPI app created with CORS")

New FastAPI app created with CORS


In [27]:
new_app.router.routes.extend(app.router.routes)

print("Existing feedback endpoints copied")

Existing feedback endpoints copied


In [28]:
import threading
import uvicorn

def run_new_server():
    uvicorn.run(new_app, host="0.0.0.0", port=8001)

server_thread = threading.Thread(target=run_new_server, daemon=True)
server_thread.start()

print("New FastAPI server started on port 8001")

New FastAPI server started on port 8001


In [29]:
from google.colab import output

public_url = output.eval_js(
    "google.colab.kernel.proxyPort(8001)"
)

print(public_url)

https://8001-m-s-kkb-use1c2-1y919o1ac5g0m-c.us-east1-2.prod.colab.dev


In [30]:
import sqlite3

conn = sqlite3.connect("feedback.db")
cursor = conn.cursor()

cursor.execute("""
SELECT id, session_id, query, response_excerpt, rating, timestamp
FROM feedback
ORDER BY id DESC
LIMIT 5
""")

for record in cursor.fetchall():
    print(record)

conn.close()

(3, 'test-session-003', 'How do I deploy my FastAPI application?', 'You can deploy FastAPI using a cloud hosting platform.', -1, '2026-09-20 08:14:56')
(2, 'test-session-002', 'Explain Python loops', 'A loop allows you to repeat a block of code multiple times.', 1, '2026-09-20 08:14:22')
(1, 'test-session-001', 'What is machine learning?', 'Machine learning is a method where computers learn patterns from data.', 1, '2026-09-20 08:11:51')


In [31]:
import sqlite3

conn = sqlite3.connect("feedback.db")
cursor = conn.cursor()

cursor.execute("""
SELECT id, query, rating, timestamp
FROM feedback
ORDER BY id DESC
LIMIT 10
""")

for record in cursor.fetchall():
    print(record)

conn.close()

(3, 'How do I deploy my FastAPI application?', -1, '2026-09-20 08:14:56')
(2, 'Explain Python loops', 1, '2026-09-20 08:14:22')
(1, 'What is machine learning?', 1, '2026-09-20 08:11:51')


In [32]:
import requests

response = requests.get("http://127.0.0.1:8001/feedback/summary")

print("Status code:", response.status_code)
print(response.json())

INFO:     127.0.0.1:55430 - "GET /feedback/summary HTTP/1.1" 200 OK
Status code: 200
{'total_interactions': 3, 'positive_rating_percentage': 66.67, 'top_5_query_topic_words': ['machine', 'learning', 'explain', 'python', 'loops'], 'three_lowest_rated_response_excerpts': [{'response_excerpt': 'You can deploy FastAPI using a cloud hosting platform.', 'rating': -1}]}


In [33]:
import sqlite3

conn = sqlite3.connect("feedback.db")
cursor = conn.cursor()

cursor.execute("""
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN rating = 1 THEN 1 ELSE 0 END) AS positive,
    SUM(CASE WHEN rating = -1 THEN 1 ELSE 0 END) AS negative
FROM feedback
""")

result = cursor.fetchone()

print("Total feedback:", result[0])
print("Positive:", result[1])
print("Negative:", result[2])

conn.close()

Total feedback: 3
Positive: 2
Negative: 1


In [34]:
import sqlite3
from collections import Counter

conn = sqlite3.connect("feedback.db")
cursor = conn.cursor()

cursor.execute("""
SELECT query, response_excerpt, rating, timestamp
FROM feedback
ORDER BY timestamp DESC
""")

records = cursor.fetchall()
conn.close()

negative_records = [r for r in records if r[2] == -1]

print("Total feedback:", len(records))
print("Negative feedback:", len(negative_records))

print("\nNegative feedback:")
for query, response, rating, timestamp in negative_records:
    print("\nQuery:", query)
    print("Response:", response)
    print("Time:", timestamp)

Total feedback: 3
Negative feedback: 1

Negative feedback:

Query: How do I deploy my FastAPI application?
Response: You can deploy FastAPI using a cloud hosting platform.
Time: 2026-09-20 08:14:56
